# Week 3 — Retrieval benchmark (dense vs BM25 vs hybrid)

Runs the fixed query set in `data/benchmarks/fiqa_sample_query_set.json` against local **fiqa_sample** index (default: first 1,000 BEIR FiQA documents).

**Prerequisites**
1. `python src/scripts/ingest.py` (builds Qdrant + BM25 pickle)
2. `.env` configured for `EMBEDDING_METHOD` / models (same as ingest)

**Outputs**
- `docs/reports/week3_retrieval_comparison.md` — comparison report
- Optional CSV under `data/benchmarks/` with raw timing rows

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

load_dotenv(REPO_ROOT / ".env")

from app.config import get_settings
from app.evaluation.retrieval_benchmark import (
    benchmark_query_set,
    load_query_set,
    render_comparison_report,
    results_to_records,
    summarize_latencies,
    summarize_overlaps,
)
from app.retrieval import Bm25ChunkIndex, QdrantChunkStore, create_embedding_provider

settings = get_settings()
if settings.hf_hub_disable_symlinks_warning:
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

QUERY_SET_PATH = REPO_ROOT / "data" / "benchmarks" / "fiqa_sample_query_set.json"
REPORT_PATH = REPO_ROOT / "docs" / "reports" / "week3_retrieval_comparison.md"
CSV_PATH = REPO_ROOT / "data" / "benchmarks" / "fiqa_sample_benchmark_runs.csv"

spec = load_query_set(QUERY_SET_PATH)
print(f"Loaded {len(spec.queries)} benchmark queries (top_k={spec.top_k})")

Loaded 10 benchmark queries (top_k=5)


In [2]:
bm25_path = settings.resolved_bm25_index_path(spec.collection)
qdrant_path = settings.resolved_qdrant_path()

if not qdrant_path or not Path(qdrant_path).exists():
    raise FileNotFoundError(
        f"Qdrant path missing: {qdrant_path}. Run: python src/scripts/ingest.py"
    )
if not bm25_path.is_file():
    raise FileNotFoundError(
        f"BM25 index missing: {bm25_path}. Re-run ingest to build the pickle."
    )

embedder = create_embedding_provider(
    settings.embedding_method,
    model_name=settings.embedding_model,
    cohere_api_key=settings.cohere_api_key,
    cohere_embed_model=settings.cohere_embed_model,
    cohere_embed_batch_size=settings.cohere_embed_batch_size,
    cohere_embed_batch_sleep_seconds=settings.cohere_embed_batch_sleep_seconds,
)
bm25_index = Bm25ChunkIndex.load(bm25_path)
store = QdrantChunkStore.from_path(qdrant_path, spec.collection)
if not store.collection_exists():
    store.close()
    raise RuntimeError(f"Collection '{spec.collection}' not found in Qdrant.")

print(f"Qdrant: {qdrant_path}")
print(f"BM25: {bm25_path}")
print(f"Embedding: {settings.embedding_method} / {settings.embedding_model}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Qdrant: D:\Repos\RAG\RAG\data\qdrant_local
BM25: D:\Repos\RAG\RAG\data\bm25_indexes\fiqa_sample.pkl
Embedding: sentence_transformer / all-MiniLM-L6-v2


In [3]:
try:
    results = benchmark_query_set(
        spec,
        store=store,
        embedder=embedder,
        bm25_index=bm25_index,
    )
finally:
    store.close()

df = pd.DataFrame(results_to_records(results))
display(df)

latency = summarize_latencies(results)
pd.DataFrame(latency).T

,query_id,theme,method,latency_ms,num_hits,top_doc_ids,top_keys
0,q01,brokerage,dense,161.4816,5,"4883,4883,4883,4883,7411","4883:0,4883:1,4883:2,4883:3,7411:0"
1,q01,brokerage,bm25,7.7673,5,"4883,4883,4883,4883,9116","4883:0,4883:1,4883:2,4883:3,9116:4"
2,q01,brokerage,hybrid,31.2248,5,"4883,4883,4883,4883,7411","4883:0,4883:1,4883:2,4883:3,7411:0"
3,q02,credit,dense,17.8900,5,"5747,8496,5747,9814,2768","2768:0,5747:2,5747:3,8496:0,9814:3"
4,q02,credit,bm25,8.5088,5,"6341,9814,4645,2018,2018","2018:1,2018:3,4645:0,6341:0,9814:0"
5,q02,credit,hybrid,24.1534,5,"6113,5747,5747,6341,8496","5747:2,5747:3,6113:1,6341:0,8496:0"
6,q03,investing,dense,16.3680,5,"5188,8266,8000,5188,9656","5188:0,5188:4,8000:0,8266:0,9656:0"
7,q03,investing,bm25,8.4151,5,"6339,680,8974,680,5188","5188:0,6339:0,680:0,680:1,8974:3"
8,q03,investing,hybrid,24.4429,5,"5188,680,5188,680,8974","5188:0,5188:4,680:0,680:1,8974:3"
9,q04,taxes,dense,15.7974,5,"7423,7423,5224,6891,5180","5180:3,5224:0,6891:0,7423:0,7423:1"


,mean_ms,p50_ms,max_ms
dense,31.18223,16.51830,161.4816
bm25,7.17513,7.12170,8.5088
hybrid,25.13345,24.29815,31.2248


In [4]:
overlaps = summarize_overlaps(results, spec.top_k)
pd.Series(overlaps, name="overlap_metrics")

dense_vs_bm25_jaccard_mean              0.181746
dense_vs_hybrid_jaccard_mean            0.495238
bm25_vs_hybrid_jaccard_mean             0.414683
hybrid_chunks_not_in_dense_topk_mean    1.900000
hybrid_chunks_not_in_bm25_topk_mean     2.200000
Name: overlap_metrics, dtype: float64

In [5]:
df.head()

,query_id,theme,method,latency_ms,num_hits,top_doc_ids,top_keys
0,q01,brokerage,dense,161.4816,5,"4883,4883,4883,4883,7411","4883:0,4883:1,4883:2,4883:3,7411:0"
1,q01,brokerage,bm25,7.7673,5,"4883,4883,4883,4883,9116","4883:0,4883:1,4883:2,4883:3,9116:4"
2,q01,brokerage,hybrid,31.2248,5,"4883,4883,4883,4883,7411","4883:0,4883:1,4883:2,4883:3,7411:0"
3,q02,credit,dense,17.8900,5,"5747,8496,5747,9814,2768","2768:0,5747:2,5747:3,8496:0,9814:3"
4,q02,credit,bm25,8.5088,5,"6341,9814,4645,2018,2018","2018:1,2018:3,4645:0,6341:0,9814:0"


In [ ]:
report_md = render_comparison_report(
    spec,
    results,
    embedding_method=settings.embedding_method,
    embedding_model=settings.embedding_model,
)

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report_md, encoding="utf-8")
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CSV_PATH, index=False)

print(f"Wrote report: {REPORT_PATH}")
print(f"Wrote CSV: {CSV_PATH}")

Wrote report: D:\Repos\RAG\RAG\docs\reports\week3_retrieval_comparison.md
Wrote CSV: D:\Repos\RAG\RAG\data\benchmarks\fiqa_sample_benchmark_runs.csv


## Spot-check one query

Use this cell to inspect ranking differences side by side after the full run.

In [6]:
spot = results[0]
print(spot.query.question, "\n")
for method, run in spot.runs.items():
    print(method.upper(), f"({run.latency_ms:.1f} ms)")
    for rank, hit in enumerate(run.hits[:3], start=1):
        print(f"  {rank}. [{hit.doc_id}:{hit.chunk_id}] {hit.chunk_text[:100]}...")
    print()

What are broker fees and how do retail brokers charge commissions? 

DENSE (161.5 ms)
  1. [4883:0] Retail brokers and are generally not members of exchanges and would generally not be members of exch...
  2. [4883:1] technology, clearing fees, regulation and human capital. Market makers will have the same costs but ...
  3. [4883:3] makers and often subtracted from POF paid to routing firms. Other (full service brokers) charging hi...

BM25 (7.8 ms)
  1. [4883:0] Retail brokers and are generally not members of exchanges and would generally not be members of exch...
  2. [4883:1] technology, clearing fees, regulation and human capital. Market makers will have the same costs but ...
  3. [4883:2] of POF has actually allowed retail brokers to keep their costs lower but to to shrinking margins and...

HYBRID (31.2 ms)
  1. [4883:0] Retail brokers and are generally not members of exchanges and would generally not be members of exch...
  2. [4883:1] technology, clearing fees, regulation and